In [5]:
!wget https://raw.githubusercontent.com/iraouiabdou/abdou_torch/main/abdou_torch.py
import abdou_torch
from abdou_torch import Tensor, Linear, MLP, AdamW, xp

--2026-07-19 15:29:21--  https://raw.githubusercontent.com/iraouiabdou/abdou_torch/main/abdou_torch.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9781 (9.6K) [text/plain]
Saving to: ‘abdou_torch.py.1’

abdou_torch.py.1    100%[===================>]   9.55K  --.-KB/s    in 0s      

2026-07-19 15:29:21 (92.8 MB/s) - ‘abdou_torch.py.1’ saved [9781/9781]



In [6]:
import tensorflow as tf

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train.shape, y_train.shape, x_test.shape, y_test.shape

((60000, 28, 28), (60000,), (10000, 28, 28), (10000,))

In [7]:
model = MLP(28*28, [1024, 512, 256, 128, 10])

In [8]:
x_train = xp.asarray(x_train.reshape(-1, 784), dtype=xp.float32) / 255.0
x_test = xp.asarray(x_test.reshape(-1, 784), dtype=xp.float32) / 255.0
y_train = xp.asarray(y_train)
y_test = xp.asarray(y_test)
epochs = 50
batch_size = 2048
steps_per_epoch = len(x_train) // batch_size
loss_history = []
optim = AdamW(model.parameters())
for epoch in range(epochs):
  epoch_loss = 0
  indices = xp.arange(len(x_train))
  xp.random.shuffle(indices)
  x_train = x_train[indices]
  y_train = y_train[indices]
  for step in range(steps_per_epoch):

    start = step * batch_size
    end = start + batch_size

    batch_x = Tensor(x_train[start:end], True)
    batch_y = y_train[start:end]

    preds = model(batch_x)

    loss = preds.crossEntropyLoss(batch_y)

    model.zero_grad()
    loss.backward()
    optim.step()
    epoch_loss += loss.data

    if step % 10 == 0:
      print(f"Epoch {epoch+1}/{epochs} | Step {step}/{steps_per_epoch} | Loss: {loss.data:.4f}")

  print(f"Epoch {epoch+1} Complete. Avg Loss: {epoch_loss/steps_per_epoch:.4f}")
  loss_history.append(epoch_loss/steps_per_epoch)


Epoch 1/50 | Step 0/29 | Loss: 2.4551
Epoch 1/50 | Step 10/29 | Loss: 0.3786
Epoch 1/50 | Step 20/29 | Loss: 0.2912
Epoch 1 Complete. Avg Loss: 0.6062
Epoch 2/50 | Step 0/29 | Loss: 0.1922
Epoch 2/50 | Step 10/29 | Loss: 0.1424
Epoch 2/50 | Step 20/29 | Loss: 0.1543
Epoch 2 Complete. Avg Loss: 0.1613
Epoch 3/50 | Step 0/29 | Loss: 0.1168
Epoch 3/50 | Step 10/29 | Loss: 0.0957
Epoch 3/50 | Step 20/29 | Loss: 0.1121
Epoch 3 Complete. Avg Loss: 0.1027
Epoch 4/50 | Step 0/29 | Loss: 0.0730
Epoch 4/50 | Step 10/29 | Loss: 0.0706
Epoch 4/50 | Step 20/29 | Loss: 0.0520
Epoch 4 Complete. Avg Loss: 0.0690
Epoch 5/50 | Step 0/29 | Loss: 0.0384
Epoch 5/50 | Step 10/29 | Loss: 0.0569
Epoch 5/50 | Step 20/29 | Loss: 0.0571
Epoch 5 Complete. Avg Loss: 0.0495
Epoch 6/50 | Step 0/29 | Loss: 0.0363
Epoch 6/50 | Step 10/29 | Loss: 0.0431
Epoch 6/50 | Step 20/29 | Loss: 0.0276
Epoch 6 Complete. Avg Loss: 0.0336
Epoch 7/50 | Step 0/29 | Loss: 0.0203
Epoch 7/50 | Step 10/29 | Loss: 0.0261
Epoch 7/50 | Step

In [9]:
correct = 0
total = 0

eval_batch_size = 1000
for i in range(0, len(x_test), eval_batch_size):
    x_batch = Tensor(x_test[i:i+eval_batch_size], requires_grad=False)

    logits = model(x_batch)

    predictions = xp.argmax(logits.data, axis=1)
    truth = y_test[i:i+eval_batch_size]

    correct += xp.sum(predictions == truth)
    total += len(truth)

acc = correct / total
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 98.28%
